# Full HH and UltraFeedback DPO: manual recipe
Fresh independent runs from full SFT best938; one epoch each, LR1e-6 fixed, beta0.1, effective batch32, context512, AdamW, clipping1.0. HH runs first; UltraFeedback follows from SFT, never HH.
The user explicitly replaced the pilot acceptance gate with direct full runs. Decoder1.05/block0/max256 is recorded consistently; response-quality evaluation remains pending.


In [ ]:
# 1. Verify both CUDA GPUs and record the starting environment.
import json, sys, subprocess
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import torch
output = Path('/kaggle/working/pa2_outputs')
output.mkdir(exist_ok=True)
info = {'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda,
        'cuda_available': torch.cuda.is_available(), 'gpu_count': torch.cuda.device_count(), 'packages': {}}
for name in ['transformers', 'datasets', 'accelerate', 'huggingface_hub', 'peft', 'trl', 'torchao']:
    try:
        info['packages'][name] = version(name)
    except PackageNotFoundError:
        info['packages'][name] = None
info['devices'] = []
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    info['devices'].append({'index': i, 'name': props.name, 'memory_gib': round(props.total_memory / 2**30, 2)})
    assert torch.ones(16, device=f'cuda:{i}').sum().item() == 16
print(json.dumps(info, indent=2))
assert info['cuda_available'] and info['gpu_count'] == 2, 'Two CUDA GPUs required'
assert all('T4' in d['name'] for d in info['devices']), 'Expected T4 devices'
# Keep the original starting environment if rerunning the setup cells.
initial = output / 'environment_before_setup.json'
if not initial.exists():
    initial.write_text(json.dumps(info, indent=2) + '\n')
print('Both T4 GPUs execute tensors.')


In [ ]:
# 2. Pin PA2 dependencies, preserve CUDA PyTorch and verify fresh Qwen imports.
torch_before = version('torch')
constraint = output / 'preserve_torch.txt'
constraint.write_text('torch==' + torch_before + '\n')
packages = ['transformers==4.57.6', 'datasets==4.8.5', 'accelerate==1.15.0', 'huggingface_hub==0.36.2',
            'tqdm', 'pyyaml', 'sentencepiece', 'safetensors', 'pytest>=8', 'matplotlib']
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                         '-c', str(constraint), *packages], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
(output / 'package_setup.log').write_text(result.stdout)
print(result.stdout[-3500:])
result.check_returncode()
assert version('torch') == torch_before, 'CUDA PyTorch must not change'
probe_command = [sys.executable, '-c', 'from transformers import AutoModelForCausalLM, AutoTokenizer; from transformers.models.qwen2.modeling_qwen2 import Qwen2ForCausalLM; print("Qwen model imports pass")']
probe = subprocess.run(probe_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
if probe.returncode and 'torchao' in probe.stdout:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    probe = subprocess.run(probe_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(probe.stdout[-3500:])
probe.check_returncode()
versions = {name: version(name) for name in ['torch', 'transformers', 'datasets', 'accelerate', 'huggingface_hub']}
(output / 'environment_after_setup.json').write_text(json.dumps(versions, indent=2) + '\n')
print(json.dumps(versions, indent=2))
print('PA2 dependencies ready. CUDA PyTorch preserved.')


In [ ]:
# Reviewed source is embedded exactly; no credential or moving notebook input.
import base64, hashlib, io, zipfile, json, sys
from pathlib import Path
archive_raw = base64.b64decode('...')
assert hashlib.sha256(archive_raw).hexdigest() == '0630ed02e196cf10acefe680bc432ba9c819515f027bd0e1b2af42ea77e35fab'
root = Path('/kaggle/working') / 'pa2-dpo-reviewed-4b7deda714e2'
assert not root.exists(), 'use a fresh session; do not overwrite source'
with zipfile.ZipFile(io.BytesIO(archive_raw)) as handle:
    assert all(n.startswith('pa2-dpo-reviewed-4b7deda714e2'+'/') and '..' not in Path(n).parts for n in handle.namelist())
    handle.extractall('/kaggle/working')
manifest_raw = (root/'source_manifest.json').read_bytes()
assert hashlib.sha256(manifest_raw).hexdigest() == '4b7deda714e222c1c0a1c10d0b0e31b85852a8828c855a8e9d3015b153d3c820'
manifest = json.loads(manifest_raw)
for name, entry in manifest['files'].items():
    raw = (root/name).read_bytes()
    assert len(raw) == entry['bytes'] and hashlib.sha256(raw).hexdigest() == entry['sha256'], name
review_raw = base64.b64decode('ewogICJzdGF0dXMiOiAicHJvdmlzaW9uYWwiLAogICJyZXZpZXdlciI6ICJDb2RleCIsCiAgImF1dGhvcml6YXRpb24iOiAidXNlciByZXF1ZXN0ZWQgZmlyc3QgRFBPIHRyYWluaW5nIGxhdW5jaCBiZWZvcmUgZW5kaW5nIHRoZSBzZXNzaW9uIiwKICAiZGV2ZWxvcG1lbnRfc2hhMjU2IjogImE3NWM2MWI0YzI5YjlmNmRiZmM4MWZkMzZlMGVmYmEyZDNiN2M2NzI5ZDkwZGRiN2RiNDU2ZDI3ODVjMDNjNzUiLAogICJyZXZpZXdlZF9pZHMiOiBbXSwKICAiY2hlY2tzIjogbnVsbCwKICAiY2FuZGlkYXRlX3NoYTI1NiI6IG51bGwsCiAgImRlY29kaW5nIjogewogICAgImRvX3NhbXBsZSI6IGZhbHNlLAogICAgIm1heF9uZXdfdG9rZW5zIjogMjU2LAogICAgInJlcGV0aXRpb25fcGVuYWx0eSI6IDEuMDUsCiAgICAibm9fcmVwZWF0X25ncmFtX3NpemUiOiAwCiAgfQp9Cg==')
assert hashlib.sha256(review_raw).hexdigest() == 'ed5c2d1140d5fe661b4278fd1e61089e37e99791d739bc79fbaebacfe7aa8f46'
review = json.loads(review_raw)
decoding_file = root/'recorded_decoding.json'
decoding_file.write_text(json.dumps(review['decoding'],sort_keys=True,indent=2)+chr(10))
(root/'decoder_review.json').write_bytes(review_raw)
sys.path.insert(0,str(root))
from src.checkpoints import verify_checkpoint
runs = list(Path('/kaggle/input').rglob('sft_full/run.json'))
assert len(runs) == 1, 'attach immutable full SFT v4 checkpoint dataset'
sft_best = runs[0].parent/'best'
origin = verify_checkpoint(sft_best)
assert origin['state']['updates'] == 938
assert origin['model_sha256'] == 'bcec9f35d309873198a727c8862638b29284cf8f6bf5cef74fddd2a9d7ee026d'
banks = list(Path('/kaggle/input').rglob('bundle_manifest.json'))
assert len(banks) == 1, 'attach prepared training data'
assert hashlib.sha256(banks[0].read_bytes()).hexdigest() == '03887a5d65a366e8ed9df3789e5aa16a56ca1a5a914da7a9277ffcc1f969363c'
prepared = banks[0].parent
print('Verified source, recorded decoder settings, prepared data and full SFT origin938',flush=True)
print('Decoder review status:',review['status'],flush=True)


In [ ]:
# Manual recipe: two independent full one-epoch DPO runs from SFT best938.
import subprocess
for track in ['hh', 'uf']:
    run_dir = Path('/kaggle/working') / ('dpo_full_' + track + '_manual_1em6')
    assert not run_dir.exists(), 'Never overwrite a prior experiment'
    command = [sys.executable, '-u', '-m', 'scripts.train_qwen', '--track', track,
               '--data-dir', str(prepared/track), '--output-dir', str(run_dir),
               '--sft-best', str(sft_best), '--full-run', '--lr', '1e-6', '--beta', '0.1',
               '--device', 'cuda:0', '--reference-device', 'cuda:1', '--eval-every', '50',
               '--development-prompts', str(root/'data/development/dpo_quality_40.json'),
               '--generation-updates', '100,200,300,400,terminal',
               '--decoding-config', str(decoding_file),
               '--exclusion-bank', str(root/'exclusion_banks/alpaca_eval.jsonl'),
               '--exclusion-bank', str(root/'eval_sets/hh_rlhf_eval_300_seed42.json')]
    print('FULL DPO TRACK:', track, 'LIVE PROGRESS:', run_dir/'metrics.jsonl',
          'STATE:', run_dir/'run.json', flush=True)
    with (root/('dpo_full_'+track+'.log')).open('x') as log:
        process = subprocess.Popen(command, cwd=root, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                log.write(line); log.flush(); print(line, end='', flush=True)
            assert process.wait() == 0, 'Full run failed; preserve evidence and diagnose'
        except BaseException:
            process.terminate(); process.wait()
            raise
    state = json.loads((run_dir/'run.json').read_text())
    assert state['status'] == 'completed' and state['updates'] == 469
    assert state['config']['max_updates'] is None and state['config']['epochs'] == 1
    assert state['metadata']['run_scope'] == 'full'
    assert state['config']['lr'] == 1e-6 and state['config']['beta'] == 0.1
    metrics = [json.loads(line) for line in (run_dir/'metrics.jsonl').read_text().splitlines()]
    validation = [r for r in metrics if r['event'] == 'validation']
    assert [r['updates'] for r in validation] == [0,50,100,150,200,250,300,350,400,450,469]
    assert validation[0]['reward_accuracy'] == 0.0
    for update in [100,200,300,400,469]:
        check = json.loads((run_dir/'development'/('update-'+str(update)+'.json')).read_text())
        assert check['status'] == 'completed' and len(check['rows']) == 40
    integrity = json.loads((run_dir/'reference_integrity.json').read_text())
    assert integrity['status'] == 'verified'
    assert integrity['initial_fingerprint'] == integrity['final_fingerprint']
    for checkpoint in ['best', 'final']:
        verified = verify_checkpoint(run_dir/checkpoint)
        assert verified['state']['metadata']['run_scope'] == 'full'
        assert verified['state']['metadata']['sft_origin']['state']['updates'] == 938
    print('FULL TRAINING COMPLETE:', track, 'evaluation pending', flush=True)
print('Both manual DPO training tracks complete; no DPO-versus-SFT quality claim yet.', flush=True)
